## 1.1 Preprocesamiento

En esta sección se preparan los datos para aplicar clustering.  
Se eliminan variables que no aportan al agrupamiento (identificadores, texto y variables categóricas no transformadas), se seleccionan variables numéricas relevantes y se estandarizan para que todas tengan la misma escala.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Cargar dataset
df = pd.read_csv("movies_2026.csv", encoding="latin1")

# Vista general
print("Dimensiones del dataset:", df.shape)
display(df.head())
print("\nColumnas:")
print(df.columns.tolist())

### Variables eliminadas y justificación

Se eliminan variables de tipo identificador, texto o categóricas no transformadas, ya que no son adecuadas para calcular distancias directamente entre observaciones.

In [ ]:
# Variables que se eliminan
columns_to_drop = [
    'id',
    'genres',
    'homePage',
    'productionCompany',
    'productionCompanyCountry',
    'productionCountry',
    'director',
    'actors',
    'actorsPopularity',
    'actorsCharacter',
    'originalTitle',
    'title',
    'originalLanguage',
    'releaseDate',
    'video'
]

# Crear copia sin esas columnas
df_clean = df.drop(columns=columns_to_drop)

print("Columnas restantes después de eliminar:")
print(df_clean.columns.tolist())
print("\nDimensiones:", df_clean.shape)

### Variables seleccionadas para clustering

Se usan variables numéricas relacionadas con presupuesto, ingresos, duración, popularidad, votos, cantidad de géneros, producción y reparto, ya que permiten comparar películas mediante distancias.

In [ ]:
# Variables numéricas que sí se usarán
numeric_features = [
    'budget',
    'revenue',
    'runtime',
    'popularity',
    'voteAvg',
    'voteCount',
    'genresAmount',
    'productionCoAmount',
    'productionCountriesAmount',
    'actorsAmount',
    'castWomenAmount',
    'castMenAmount',
    'releaseYear'
]

# Subconjunto final para clustering
df_clust = df_clean[numeric_features].copy()

# Revisar nulos
print("Nulos por columna:")
print(df_clust.isnull().sum())

# Eliminar filas con nulos
df_clust = df_clust.dropna()

print("\nDimensiones después de quitar nulos:", df_clust.shape)
display(df_clust.head())

In [ ]:
# Escalado 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clust)

# Convertir a DataFrame para visualizar mejor
X_scaled_df = pd.DataFrame(X_scaled, columns=numeric_features)

print("Datos estandarizados (primeras filas):")
display(X_scaled_df.head())

Se eliminaron variables de texto, identificadores y categóricas no transformadas porque no aportan a la comparación numérica entre películas.  
Luego se seleccionaron variables numéricas relevantes para el análisis y se estandarizaron para evitar que variables con valores grandes (como budget o revenue) dominen el agrupamiento.

## 1.2 Tendencia al agrupamiento

Antes de aplicar clustering, se evalúa si los datos tienen una estructura agrupable.  
Para ello se usa el estadístico de Hopkins y una visualización tipo VAT.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from random import sample
from numpy.random import uniform

def hopkins_stat(X):
    X = np.array(X)
    n, d = X.shape
    m = max(10, int(0.1 * n))  

    nbrs = NearestNeighbors(n_neighbors=2).fit(X)

    # Puntos aleatorios dentro del rango de los datos
    rand_X = uniform(np.min(X, axis=0), np.max(X, axis=0), (m, d))

    u_dist = []
    w_dist = []

    random_indices = sample(range(n), m)

    for j in range(m):
        # Distancia de punto aleatorio al vecino más cercano real
        u = nbrs.kneighbors([rand_X[j]], n_neighbors=2, return_distance=True)[0][0][0]
        u_dist.append(u)

        # Distancia de punto real a su vecino real más cercano (ignorando sí mismo)
        w = nbrs.kneighbors([X[random_indices[j]]], n_neighbors=2, return_distance=True)[0][0][1]
        w_dist.append(w)

    H = sum(u_dist) / (sum(u_dist) + sum(w_dist))
    return H

hopkins_value = hopkins_stat(X_scaled)
print(f"Hopkins: {hopkins_value:.4f}")

### Interpretación de Hopkins

- Valor cercano a 0.5: datos con estructura aleatoria
- Valor mayor a 0.7: buena tendencia al agrupamiento
- Valor cercano a 1: fuerte tendencia al agrupamiento


In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, leaves_list

# Matriz de distancias entre observaciones
D = squareform(pdist(X_scaled, metric='euclidean'))

# Reordenamiento jerárquico para aproximar VAT
Z = linkage(X_scaled, method='ward')
order = leaves_list(Z)

D_reordered = D[order][:, order]

plt.figure(figsize=(7, 6))
plt.imshow(D_reordered, aspect='auto')
plt.title("VAT (Visual Assessment of Tendency)")
plt.xlabel("Observaciones reordenadas")
plt.ylabel("Observaciones reordenadas")
plt.colorbar()
plt.show()

### Interpretación del VAT

En el gráfico VAT se observan bloques o zonas más oscuras, lo que sugiere la presencia de grupos naturales en los datos.  
Si la imagen se ve totalmente uniforme, la tendencia al agrupamiento sería débil.

## 1.3 Número de clústeres

Se utiliza el método del codo para estimar el número adecuado de clústeres.  
Este método grafica la inercia para distintos valores de k y permite identificar un punto de inflexión.

In [ ]:
from sklearn.cluster import KMeans

inertia = []
K_values = range(1, 11)

for k in K_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(7, 5))
plt.plot(K_values, inertia, marker='o')
plt.xticks(K_values)
plt.xlabel("Número de clústeres (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.grid(True)
plt.show()

### Justificación del número de clústeres

En la gráfica del codo se identifica un punto de inflexión en **k = X** (reemplazar con el valor observado).  
A partir de ese valor, la reducción de inercia es menor, por lo que se considera una cantidad adecuada de clústeres para el análisis.

pd.DataFrame({
    "k": list(K_values),
    "inercia": inertia
})

k-means

In [ ]:
from sklearn.cluster import KMeans

k = 4

kmeans = KMeans(n_clusters=k, random_state=42)
clusters_kmeans = kmeans.fit_predict(X_scaled)

df_cluster['cluster_kmeans'] = clusters_kmeans

clustering jerarquico 

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster

Z = linkage(X_scaled, method='ward')

clusters_jer = fcluster(Z, k, criterion='maxclust')

df_cluster['cluster_jerarquico'] = clusters_jer

Comparación básica de tamaños de clusters

In [ ]:
print("KMeans")
print(df_cluster['cluster_kmeans'].value_counts())

print("\nJerárquico")
print(df_cluster['cluster_jerarquico'].value_counts())

Calidad del agrupamiento

Silhouette para K-Means

In [ ]:
from sklearn.metrics import silhouette_score

sil_kmeans = silhouette_score(X_scaled, clusters_kmeans)
print("Silhouette KMeans:", sil_kmeans)

Silhouette para Jerárquico

In [ ]:
sil_jer = silhouette_score(X_scaled, clusters_jer)
print("Silhouette Jerárquico:", sil_jer)

Gráfica comparativa

In [ ]:
plt.bar(['KMeans', 'Jerárquico'], [sil_kmeans, sil_jer])
plt.ylabel("Silhouette Score")
plt.title("Comparación de Calidad de Clustering")
plt.show()

Interpretación de los Grupos

Medidas de tendencia central

In [ ]:
df_cluster.groupby('cluster_kmeans').mean()

Variables categóricas

In [ ]:
pd.crosstab(df['originalLanguage'], df_cluster['cluster_kmeans'])